Autre implementation:
https://github.com/stephenbeckr/L-BFGS-B-C/blob/master/src/subalgorithms.c

L'intégralité du code est facile à comprendre. Il n'y a qu'un point un peu plus compliqué, et nous le détaillons ici:

Section 5 of Byrd et al. [1995] describes three methods for performing the subspace minimization: direct primal, primal CG, and dual. 
As explained by Zhu et al. (1997), "primal and dual can be implemented in a unified framework in which they are very similar; they require essentially the same amount of computation and perform equally well in practice". As a consequence, the current L-BFGS-B code (887) uses only the primal method for subspace minimization. However, the resolution differs a bit from the original paper (recipe given in pages 11-12). Indeed, Zhu et al. 1997, Zhu et al (1997) explain that in eq . 5.11

$$
    \hat{\mathbf{d}}^{u} = \dfrac{1}{\theta}\hat{\mathbf{r}}^{c}+\dfrac{1}{\theta^{2}}\mathbf{Z}^{\mathrm{T}}\mathbf{Z}\left(\mathbf{I} - \dfrac{1}{\theta}\mathbf{MW}^{\mathrm{T}}\mathbf{ZZ}^{\mathrm{T}}\mathbf{W}\right)^{-1}\mathbf{M} \mathbf{W}^{\mathrm{T}}\mathbf{Z}\hat{\mathbf{r}}^{c}
$$

Can actually be written as 

$$
    \hat{\mathbf{d}}^{u} = \dfrac{1}{\theta}\hat{\mathbf{r}}^{c}+\dfrac{1}{\theta^{2}}\mathbf{Z}^{\mathrm{T}}\mathbf{Z}\mathbf{K}^{-1} \mathbf{W}^{\mathrm{T}}\mathbf{Z}\hat{\mathbf{r}}^{c}
$$

where 

$$
    \mathbf{K}^{-1} = (\mathbf{I} - \dfrac{1}{\theta}\mathbf{MW}^{\mathrm{T}}\mathbf{ZZ}^{\mathrm{T}}\mathbf{W})^{-1}\mathbf{M} 

$$

and they remark that

$$
    \mathbf{K} = \begin{bmatrix} -\mathbf{D} - \dfrac{1}{\theta}\mathbf{Y}^{\mathrm{T}}\mathbf{ZZ}^{\mathrm{T}}\mathbf{Y}  & \mathbf{L}_{A}^{\mathrm{T}} - \mathbf{R}_{Z}^{\mathrm{T}} \\ \mathbf{L}_{A} - \mathbf{R}_{Z} & \theta \mathbf{S}^{\mathrm{T}}\mathbf{AA}^{\mathrm{T}}\mathbf{S} \end{bmatrix}
$$

where $\mathbf{L}_A$ is the strict lower triangle of $\mathbf{S}^{\mathrm{T}}\mathbf{AA}^{\mathrm{T}}\mathbf{S}$ and $\mathbf{R}_{Z}$ is the upper triangle of $\mathbf{Y}^{\mathrm{T}}\mathbf{ZZ}^{\mathrm{T}}\mathbf{Y}$. Although this matrix is not positive definite, they claim it can be factorized symmetrically by using Cholesky factorizations of the submatrices, and we do so in the L-BFGS-B code. However, the factorization is not explained. Thus, we would like to provide some details about how to achieve it. Recall the $\mathbf{LDL}^{\mathrm{T}}$ factorization for a block matrix:

$$\mathbf{K} = \begin{pmatrix} \mathbf{K}_{11} & \mathbf{K}_{21}^{\mathrm{T}} \\ \mathbf{K}_{21} & \mathbf{K}_{22} \end{pmatrix}
 =\begin{pmatrix} 1 & 0 \\ \mathbf{K}_{21} \mathbf{K}_{11}^{-1} & 1 \end{pmatrix}
  \begin{pmatrix} \mathbf{K}_{11} & 0 \\ 0 & \mathbf{S} \end{pmatrix}
  \begin{pmatrix} 1 & \mathbf{K}_{11}^{-1} \mathbf{K}_{21}^{\mathrm{T}} \\ 0 & 1 \end{pmatrix}$$
where $\mathbf{S} = \mathbf{K}_{22} - \mathbf{K}_{21} \mathbf{K}_{11}^{-1} \mathbf{K}_{21}^{\mathrm{T}}$ is the Schur complement. Clearly you need $\mathbf{K}_{11}^{-1}$ and $\mathrm{S}^{-1}$ to solve using this factorization, with $\mathbf{K}_{22}^{-1}$ providing no value.

Substituting $\mathbf{K}_{11} = \mathbf{L}_{11} \mathbf{L}_{11}^{\mathrm{T}}$ and $\mathbf{S} = \mathbf{L}_S \mathbf{L}_S^{\mathrm{T}}$ into the factorization above, we obtain the Cholesky factorization [1]:
$$\begin{aligned}
\mathbf{K} & =\begin{pmatrix}
\mathbf{K}_{11} & \mathbf{K}_{21}^{\mathrm{T}}\\
\mathbf{K}_{21} & \mathbf{K}_{22}
\end{pmatrix}\\
 & =\begin{pmatrix}
\mathbb{1} & 0\\
\mathbf{K}_{21} \left( \mathbf{L}_{11} \mathbf{L}_{11}^{\mathrm{T}}\right)^{-1} & \mathbb{1}
\end{pmatrix}\begin{pmatrix}
\mathbf{L}_{11} \mathbf{L}_{11}^{\mathrm{T}} & 0\\
0 & \mathbf{L}_S \mathbf{L}_S^{\mathrm{T}}
\end{pmatrix}\begin{pmatrix}
\mathbb{1} & \left( \mathbf{L}_{11} \mathbf{L}_{11}^{\mathrm{T}}\right)^{-1} \mathbf{K}_{21}^{\mathrm{T}}\\
0 & \mathbb{1}
\end{pmatrix}\\
 & =\left(\begin{pmatrix}
\mathbb{1} & 0\\
\mathbf{K}_{21}\left( \mathbf{L}_{11} \mathbf{L}_{11}^{\mathrm{T}}\right)^{-1} & \mathbb{1}
\end{pmatrix}\begin{pmatrix}
\mathbf{L}_{11} & 0\\
0 & \mathbf{L}_S
\end{pmatrix}\right)\left(\begin{pmatrix}
\mathbf{L}_{11}^{\mathrm{T}} & 0\\
0 & \mathbf{L}_S^{\mathrm{T}}
\end{pmatrix}\begin{pmatrix}
\mathbb{1} & \left( \mathbf{L}_{11} \mathbf{L}_{11}^{\mathrm{T}}\right)^{-1} \mathbf{K}_{21}^{\mathrm{T}}\\
0 & \mathbb{1}
\end{pmatrix}\right)\\
 & =\begin{pmatrix}
\mathbf{L}_{11} & 0\\
\mathbf{K}_{21}\mathbf{L}_{11}^{-T} & \mathbf{L}_S
\end{pmatrix}\begin{pmatrix}
\mathbf{L}_{11}^{\mathrm{T}} & \mathbf{L}_{11}^{-1} \mathbf{K}_{21}^{\mathrm{T}}\\
0 & \mathbf{L}_S^{\mathrm{T}}
\end{pmatrix} = \mathbf{L}_{K} \mathbf{L}_{K}^{\mathrm{T}} .
\end{aligned}
$$

In total obtaining $\mathbf{L}_{K}$ requires 4 factorizations.

[1] Note that this "block Cholesky" factorization is also valid with $\mathbf{L}_{11}$ replaced by a $\mathbf{K}_{11}^{1/2}$ or another non-triangular factorization of $\mathbf{K}_{11}$.